# Multi-agent research pipeline on a Kaggle GPU — Gemma 4 (4-bit GGUF)

Runs the full six-agent pipeline (Literature → Hypothesis → Experiment Planner → Coder → Writer ⇄ Reviewer)
end to end inside a Kaggle notebook, driven by a **4-bit quantized Gemma 4** served locally by `llama-server`.

**No pipeline code changes are involved.** `llama-server` speaks the OpenAI API, so this is the repo's
existing `LLM_BACKEND=openai` path pointed at `127.0.0.1` instead of a vLLM node — the same path
`scripts/slurm/run_pipeline.sbatch` uses on Barkla. Everything Kaggle-specific lives in
`scripts/kaggle/gguf_server.py` (build, download, launch, health-check) and in the environment
variables set below.

### Model

[`unsloth/gemma-4-12b-it-GGUF`](https://huggingface.co/unsloth/gemma-4-12b-it-GGUF) at **`Q4_K_M`** — the
plain 4-bit K-quant, ~7GB of weights. That size is the whole reason 4-bit matters here: BF16 would be ~24GB
and wouldn't load on a single Kaggle card at all.

unsloth's "Dynamic 2.0" 4-bit in the same repo is `UD-Q4_K_XL` — marginally larger, quantized per-layer,
usually a little more accurate. Either works; set `QUANT` below.

**VRAM.** Weights are fixed at ~7GB, so `CTX_SIZE` is what decides whether the model loads:
the KV cache grows with it. `24576` is sized to clear the widest prompt any agent builds
(~4k tokens of batched paper abstracts) plus a full 8192-token completion, and fits a single 16GB T4.
On **T4 x2** llama.cpp spreads the layers over both cards automatically and you have room to raise it.

### Kaggle settings this notebook requires

| Setting | Value | Why |
|---|---|---|
| **Accelerator** | GPU T4 x2 (P100 / L4 also fine) | A CPU build makes a run take hours; two cards give the 12B headroom |
| **Internet** | **On** | arXiv, Semantic Scholar, Hugging Face, PyPI, GitHub |
| **Persistence** | Files only (optional) | Keeps `/kaggle/working` outputs between sessions |

Expect roughly **10–15 min** of one-time setup (llama.cpp CUDA build, then a ~7GB download) before the
pipeline starts.

## 0. Configuration

Everything you'd normally want to change lives in this one cell.

In [ ]:
# --- model -----------------------------------------------------------------
REPO_ID  = "unsloth/gemma-4-12b-it-GGUF"
QUANT    = "Q4_K_M"         # 4-bit, ~7GB. Or "UD-Q4_K_XL" for unsloth Dynamic 2.0
CTX_SIZE = 24576            # must exceed the largest agent prompt + LLM_MAX_TOKENS,
                            # and is the main driver of VRAM beyond the weights

# --- the run ---------------------------------------------------------------
RESEARCH_QUESTION = "How effective is cross-lingual transfer learning (e.g., using mBERT) for Named Entity Recognition (NER) in endangered or low-resource languages?"
MAX_RESULTS_PER_QUERY = 5   # papers per generated query, per source (arXiv + Semantic Scholar)
MAX_ITERATIONS        = 2   # Writer/Reviewer cycles; each one is a full paper redraft
QUALITY_THRESHOLD     = 4   # 1-5, the Reviewer's per-dimension pass mark

# --- where the code comes from ---------------------------------------------
# "git"     : clone from a GitHub remote (set REPO_URL)
# "dataset" : you uploaded the repo as a Kaggle Dataset (set REPO_PATH to its dir)
REPO_SOURCE = "dataset"
REPO_URL    = "https://github.com/<you>/<this-repo>.git"
REPO_PATH   = "/kaggle/input/datasets/yashshere/multi-agent-research-pipeline-src"

# --- optional --------------------------------------------------------------
# Add via Kaggle's "Secrets" panel (Add-ons -> Secrets) rather than pasting keys here.
SEMANTIC_SCHOLAR_SECRET = "SEMANTIC_SCHOLAR_API_KEY"   # skipped, with a warning, if unset

WORK_DIR    = "/kaggle/working"     # outputs land here and become notebook output
SCRATCH_DIR = "/kaggle/temp/llm"    # llama.cpp build + GGUF weights (not committed)


## 1. Probe the machine

Check the accelerator, CUDA toolkit and network *before* spending ten minutes on a build.

In [ ]:
import shutil, subprocess, sys
from pathlib import Path

print("GPU:")
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout or "  none detected")
print("nvcc:  ", shutil.which("nvcc") or "MISSING - CUDA build will fail")
print("cmake: ", shutil.which("cmake") or "MISSING - installed in the next cell")
print("disk free: %.1f GB" % (shutil.disk_usage("/").free / 1e9))

def _tree(p, depth=0):
    if depth > 3 or not p.exists():
        return
    for child in sorted(p.iterdir()):
        print("  " * depth + str(child))
        if child.is_dir():
            _tree(child, depth + 1)

print("/kaggle/input tree:")
_tree(Path("/kaggle/input"))

import socket
try:
    socket.create_connection(("huggingface.co", 443), timeout=5).close()
    print("internet: ok")
except OSError:
    print("internet: BLOCKED - turn Internet on in the notebook Settings panel and re-run")


## 2. Get the pipeline and install it

Kaggle has no `uv` and the repo isn't on PyPI, so this installs the package from source with pip.
`uv` is installed too — not for the install, but because the **Coder Agent uses `uv venv` at runtime** to
build an isolated environment whenever generated experiment code needs a package that isn't already
importable. Without it on `PATH`, those experiments come back as `code_generated_not_run` with a
"'uv' is not on PATH" reason instead of results.

In [ ]:
import importlib, os, shutil, subprocess, sys
from pathlib import Path

if REPO_SOURCE == "git":
    repo_dir = Path("/kaggle/working/research-pipeline")
    if not repo_dir.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(repo_dir)], check=True)
else:
    # /kaggle/input is read-only, and an editable install needs to write egg-info.
    src = Path(REPO_PATH)
    repo_dir = Path("/kaggle/working/research-pipeline")
    if not repo_dir.exists():
        shutil.copytree(src, repo_dir)

print("repo:", repo_dir)
assert (repo_dir / "pyproject.toml").exists(), f"no pyproject.toml under {repo_dir}"

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(repo_dir)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "cmake", "huggingface_hub", "uv"], check=True)

# pip installs console scripts next to the interpreter; the Coder Agent looks for
# `uv` on PATH, which in a notebook kernel doesn't include that directory.
os.environ["PATH"] = f"{Path(sys.executable).parent}:{os.environ['PATH']}"
print("uv:", shutil.which("uv"))

# An editable install is a .pth/finder file that Python's site module only
# processes at interpreter startup - a plain `import` right after `pip install -e`
# in the SAME live kernel won't see it (ModuleNotFoundError) until the kernel
# restarts. Adding src/ to sys.path directly sidesteps that timing problem
# entirely, so this notebook never needs a restart mid-run. This is the one
# path that matters here - pip's own dependency installs (langgraph, langchain,
# etc.) land in site-packages, which is already on sys.path.
sys.path.insert(0, str(repo_dir / "src"))
importlib.invalidate_caches()

import research_pipeline
print("research_pipeline:", research_pipeline.__file__)

## 3. Build llama.cpp (CUDA) and download the weights

llama.cpp publishes no prebuilt **Linux CUDA** binary — the CUDA archives are Windows-only — so this
compiles from source. Two narrowings keep it to a few minutes rather than the best part of an hour:
only the `llama-server` target is built, and only for this machine's own compute capability
(`7.5` on a T4), probed rather than assumed.

Then ~7GB of 4-bit weights come down from Hugging Face. The build tree and the GGUF go to `/kaggle/temp`,
not `/kaggle/working`: together they're ~10GB, and anything under `working` is packaged as notebook output
on commit.

In [ ]:
import logging, sys
from pathlib import Path

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s", force=True)
sys.path.insert(0, str(repo_dir / "scripts" / "kaggle"))

import gguf_server

server = gguf_server.serve(
    repo_id=REPO_ID,
    quant=QUANT,
    install_root=SCRATCH_DIR,
    ctx_size=CTX_SIZE,
)

print("\nready")
print("  base_url:", server.base_url)
print("  model   :", server.model_alias)
print("  weights :", server.model_path)
print("  log     :", server.log_path)

### 3b. Smoke-test the endpoint

One raw request, before any agent depends on it. If this fails, the problem is the server, not the pipeline.

In [ ]:
import json, urllib.request

payload = {
    "model": server.model_alias,
    "messages": [{"role": "user", "content": 'Reply with only this JSON: {"ok": true}'}],
    "temperature": 0.0,
    "max_tokens": 32,
}
request = urllib.request.Request(
    f"{server.base_url}/chat/completions",
    data=json.dumps(payload).encode(),
    headers={"Content-Type": "application/json"},
)
with urllib.request.urlopen(request, timeout=120) as response:
    body = json.loads(response.read())

print(body["choices"][0]["message"]["content"])
print("tokens:", body.get("usage"))

### 3c. (Optional) Expose this server for a locally-run frontend

Skip this cell if you're running the rest of this notebook end to end (section 5 below) — it's only
needed to use Kaggle as *just* the LLM backend while driving the pipeline from `research-pipeline serve`
(the browser UI) on your own machine, since Kaggle accepts no inbound connections and `server.base_url`
above is loopback-only from here.

This opens a [Cloudflare quick tunnel](https://developers.cloudflare.com/cloudflare-one/connections/connect-networks/do-more-with-tunnels/trycloudflare/)
— no account or token needed — that proxies a random `https://*.trycloudflare.com` URL back to
`server.base_url`. That URL has **no authentication**: anyone who has it can call the model for as long
as this cell's tunnel process and the Kaggle session stay up, so don't leave it running longer than you
need it, and don't paste it anywhere public.

On your own machine, point the pipeline at it instead of building/serving anything locally:

```bash
uv sync --extra webapp
LLM_BACKEND=openai LLM_BASE_URL=<printed-url>/v1 LLM_MODEL=gemma-4-local LLM_API_KEY=not-needed \
  uv run research-pipeline serve
```

Then open <http://127.0.0.1:8000> and start a run — every LLM call travels over this tunnel to the GPU
here, while arXiv/Semantic Scholar are queried directly from your machine as usual.

In [ ]:
import tunnel

llm_tunnel = tunnel.start_tunnel(gguf_server.DEFAULT_PORT)
print("public URL:", llm_tunnel.public_url)
print("  point LLM_BASE_URL at:", f"{llm_tunnel.public_url}/v1")
print("log:", llm_tunnel.log_path)

## 4. Configure the pipeline

**Order matters here.** `research_pipeline.config` reads the environment at *import* time and freezes it
into a module-level `settings` object, so every variable has to be set before the first
`import research_pipeline`. If you re-run this cell after importing, restart the kernel — editing
`os.environ` afterwards changes nothing.

`LLM_MODEL` must match the name the server advertises at `/v1/models`, which is the `--alias`
`gguf_server` passes, not the GGUF filename.

In [ ]:
import os
from pathlib import Path

os.chdir(WORK_DIR)   # relative output dirs resolve here

# --- model endpoint --------------------------------------------------------
os.environ["LLM_BACKEND"]  = "openai"       # llama-server is OpenAI-compatible
os.environ["LLM_BASE_URL"] = server.base_url
os.environ["LLM_MODEL"]    = server.model_alias
os.environ["LLM_API_KEY"]  = "not-needed"   # llama-server doesn't check it

# Low temperature: every agent parses structured JSON out of the response, and a
# 4-bit model has less headroom for well-formed output than the BF16 Nemotron
# this pipeline was tuned against. llm_json still repairs one bad parse per call.
os.environ["LLM_TEMPERATURE"] = "0.2"
os.environ["LLM_TOP_P"]       = "0.95"
# Truncation is a hard failure (a half-written Writer section never parses), so
# leave room — but keep prompt + completion inside CTX_SIZE above.
os.environ["LLM_MAX_TOKENS"]     = "8192"
os.environ["LLM_ENABLE_THINKING"] = "false"

# --- optional API key ------------------------------------------------------
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["SEMANTIC_SCHOLAR_API_KEY"] = UserSecretsClient().get_secret(SEMANTIC_SCHOLAR_SECRET)
    print("Semantic Scholar key loaded from Kaggle Secrets")
except Exception as exc:
    print(f"No Semantic Scholar key ({exc.__class__.__name__}) - that source is skipped, arXiv still runs")

# --- where artifacts go ----------------------------------------------------
outputs = Path(WORK_DIR) / "outputs"
for var in ("HYPOTHESIS_OUTPUT_DIR", "EXPERIMENT_PLANNER_OUTPUT_DIR",
            "CODER_OUTPUT_DIR", "WRITER_OUTPUT_DIR", "REVIEWER_OUTPUT_DIR"):
    os.environ[var] = str(outputs)
os.environ["WRITER_REVIEWER_LOOP_OUTPUT_DIR"] = str(outputs / "paper")
os.environ["CODER_EXPERIMENTS_DIR"] = str(Path(WORK_DIR) / "experiments")

# There is no SLURM here, so nothing may try to submit a job. This is already the
# default; set explicitly because it's the one setting that would fail loudly on Kaggle.
os.environ["CODER_AUTO_SUBMIT_SLURM"] = "false"

os.environ["WRITER_REVIEWER_MAX_ITERATIONS"]   = str(MAX_ITERATIONS)
os.environ["WRITER_REVIEWER_QUALITY_THRESHOLD"] = str(QUALITY_THRESHOLD)

print("configured against", os.environ["LLM_BASE_URL"])

## 5. Run the pipeline

One `graph.invoke` — the same call `research-pipeline orchestrate` makes. Stages stream as they complete
rather than reporting only at the end, so a long run is observable.

**What to expect on Kaggle.** A GPU *is* present, so the Coder Agent runs `low`/`medium` complexity
experiments synchronously, including ones that self-report `needs_gpu` — the probe is at runtime, not
hardcoded. `high` complexity plans still get code plus a `run.sbatch` they'll never be submitted with
here; that file is a harmless artifact on a machine with no scheduler.

In [ ]:
import logging, time, uuid

logging.getLogger("research_pipeline").setLevel(logging.INFO)

from research_pipeline.orchestrator.graph import build_pipeline_graph

graph = build_pipeline_graph()
initial_state = {
    "research_question": RESEARCH_QUESTION,
    "max_results_per_query": MAX_RESULTS_PER_QUERY,
    "download_dir": f"{WORK_DIR}/papers",
    "metadata_path": f"{WORK_DIR}/papers/metadata.json",
    "output_dir": str(outputs),
    "max_iterations": MAX_ITERATIONS,
    "quality_threshold": QUALITY_THRESHOLD,
}
run_config = {"configurable": {"thread_id": str(uuid.uuid4())}, "recursion_limit": 50}

started = time.time()
final_state = {}
for update in graph.stream(initial_state, config=run_config, stream_mode="updates"):
    for node, delta in update.items():
        print(f"[{time.time() - started:7.1f}s] {node}: {sorted(delta)}")
        final_state.update(delta)

print(f"\ndone in {(time.time() - started) / 60:.1f} min")

## 6. Results

In [ ]:
result = final_state["final_result"]

print("final paper   :", result["final_paper_path"])
print("iterations run:", result["iterations_run"])
print("converged     :", result["converged"])
print("review history:", result["review_history_path"])
for issue in result["unresolved_issues"]:
    print("  unresolved:", issue)

# Per-experiment outcomes: "completed" ran here, "code_generated_not_run" did not.
for experiment in final_state.get("coder_output", {}).get("experiments", []):
    print(f"  {experiment['hypothesis_id']}: {experiment['status']} {experiment.get('reason', '')}".rstrip())

### Keep the artifacts

Everything already lives under `/kaggle/working`, so **Save Version** captures it. This bundles the run into
one archive to download instead of clicking through the output tree — the PDF, every review iteration, the
generated experiment code, and the downloaded papers.

In [ ]:
import shutil
from pathlib import Path

archive = shutil.make_archive(f"{WORK_DIR}/run_artifacts", "zip", root_dir=WORK_DIR, base_dir="outputs")
print(archive, f"({Path(archive).stat().st_size / 1e6:.1f} MB)")

# FileLink, not an embedded IFrame: Jupyter resolves embeds relative to the
# notebook's serving root, and these paths are absolute.
from IPython.display import FileLink, display
display(FileLink(Path(archive).relative_to(WORK_DIR).as_posix()))
display(FileLink(Path(result["final_paper_path"]).resolve().relative_to(WORK_DIR).as_posix()))

## 7. Shut the server down

Frees the VRAM. Do this before running anything else GPU-heavy in the same session.

In [ ]:
if "llm_tunnel" in dir():
    tunnel.stop_tunnel(llm_tunnel)
    print("tunnel stopped")

gguf_server.stop(server)
print("server stopped")

## Troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| `No CUDA GPU detected` | Accelerator is "None" | Settings → Accelerator → GPU T4 x2, then restart the session |
| `ModuleNotFoundError: No module named 'research_pipeline'` in cell 5 | An editable install's `.pth` finder is only processed by Python's `site` module at interpreter startup — a live kernel doesn't pick it up mid-session | Already handled — cell 2 adds `repo_dir/src` to `sys.path` directly instead of relying on the install to be re-scanned, and smoke-tests the import right there. If it still happens, cell 2 didn't actually run before cell 5 — re-run it |
| `nvcc is not on PATH` | CPU-only image | Same as above — the CUDA toolkit ships with Kaggle's GPU images |
| `llama-server exited ... unknown model architecture` | llama.cpp predates Gemma 4 | `gguf_server.build_llama_server(SCRATCH_DIR, force=True)` to re-clone master |
| `CUDA out of memory` while loading | 7GB of weights + KV cache exceeds one card | Switch to T4 x2, lower `CTX_SIZE`, or pass `extra_args=["--cache-type-k", "q8_0", "--cache-type-v", "q8_0"]` to `serve()` |
| `cmake configure failed` ... `CUDA::cuda_driver ... target was not found` | Kaggle's container mounts the real GPU driver as `libcuda.so.1` with no unversioned `libcuda.so`, which is the one file CMake's `find_library()` needs to create `CUDA::cuda_driver` at all | Already handled — `gguf_server.build_llama_server` locates the toolkit's link-only stub and stages it as both `libcuda.so`/`libcuda.so.1` before configuring. If it still fails, the log line above it says `Could not locate the CUDA toolkit's libcuda.so stub`; the toolkit install is missing its `lib64/stubs` (or `targets/*/lib/stubs`) directory entirely — a devel, not runtime-only, CUDA toolkit image is required |
| Download fills the disk | Both build tree and weights on the same volume | Check free space in cell 1; `SCRATCH_DIR` needs ~10GB |
| `LLMJSONError` after the repair retry | 4-bit model drifting off-format | Lower `LLM_TEMPERATURE` to `0.0`, or move up to `Q5_K_M`/`Q6_K` |
| Truncated section, malformed JSON on long prompts | Prompt + completion exceeded `CTX_SIZE` | Raise `CTX_SIZE`, or lower `MAX_RESULTS_PER_QUERY` |
| Semantic Scholar warning at startup | No API key | Expected — arXiv still runs; add the key via Add-ons → Secrets |
| `'uv' is not on PATH` in an experiment `reason` | Kernel `PATH` missing pip's script dir | Re-run cell 2, which prepends it |
| Run outlives the session | Kaggle caps a GPU session at 9h (12h/week free quota) | Lower `MAX_ITERATIONS` / `MAX_RESULTS_PER_QUERY`, or split the work |

### Running it from a terminal instead

`gguf_server.py` is a standalone script — nothing in it imports `research_pipeline`. To hold a server open
and drive the CLI against it:

```bash
python scripts/kaggle/gguf_server.py --foreground &
LLM_BACKEND=openai LLM_BASE_URL=http://127.0.0.1:8000/v1 LLM_MODEL=gemma-4-local \
  research-pipeline orchestrate "your research question"
```